In [0]:
## Do this over DLT pipeline, not in notebook

# Just copy paste cell into single python file

In [0]:
import dlt
from pyspark.sql.functions import col, lit, current_timestamp

In [0]:
username = dbutils.secrets.get(scope="sqlserver-creds", key="username")
print ("username", username) # READACTED 

password = dbutils.secrets.get(scope="sqlserver-creds", key="password")
print ("password", password)  # READACTED

In [0]:
DATABASE = "free-sql-db-5202703"
HOSTNAME = "gks-test.database.windows.net"

# f-string

jdbc_url = f"jdbc:sqlserver://{HOSTNAME}:1433;database={DATABASE};encrypt=true;trustServerCertificate=false;hostNameInCertificate=*.database.windows.net;loginTimeout=30;"

jdbc_properties = {
    "user": username ,
    "password": password ,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

In [0]:
@dlt.table(name="products_cdc_raw")
def read_cdc_from_sql():
    return spark.read.jdbc(
        url=jdbc_url,
        table="cdc.gks_products_CT",
        properties=jdbc_properties
    )

Name,Type
__$start_lsn,binary
__$end_lsn,binary
__$seqval,binary
__$operation,int
__$update_mask,binary
id,int
name,string
price,"decimal(10,2)"
stock,int
__$command_id,int


In [0]:
# Step 2: Filter only insert/update rows, and add last_updated timestamp
@dlt.table
def products_cdc_clean():
    return (
        dlt.read("products_cdc_raw")
        .filter(col("__$operation").isin(2, 4))  # 2 = insert, 4 = update after
        .select(
            col("id"),
            col("name"),
            col("price"),
            col("stock"),
            current_timestamp().alias("last_updated")
        )
    )

Name,Type
id,int
name,string
price,"decimal(10,2)"
stock,int
last_updated,timestamp


In [0]:

dlt.create_target_table('products_scd1', 
                        table_properties = {'delta.enableChangeDataFeed': 'true'})

# Step 3: Apply SCD Type 1 using apply_changes (merge by id)
dlt.apply_changes(
    target = "products_scd1",
    source = "products_cdc_clean",
    keys = ["id"],
    sequence_by = col("last_updated"),
    
    except_column_list = [],   # All columns are overwritten
    stored_as_scd_type = 1
)

/databricks/spark/python/dlt/helpers.py:139: UserWarning: Function create_target_table has been deprecated. Please use create_streaming_table instead.
  warnings.warn("Function {0} has been deprecated. Please use {1} instead."


Name,Type
id,int
name,string
price,"decimal(10,2)"
stock,int
last_updated,timestamp


In [0]:
# Step 2: Filter only insert/update rows, and add last_updated timestamp
# SCD 2 maintain history of changes
@dlt.table
def products_cdc_clean_scd2():
    return (
        dlt.read("products_cdc_raw")
        .filter(col("__$operation").isin(2, 4))  # 2 = insert, 4 = update after
        .select(
            col("id"),
            col("name"),
            col("price"),
            col("stock"),
            col("__$seqval").alias("sequenceNum")
        )
    ) 


dlt.create_target_table('products_scd2', 
                        table_properties = {'delta.enableChangeDataFeed': 'true'})

dlt.apply_changes(
    target="products_scd2",
    source="products_cdc_clean_scd2",
    keys=["id"],
    sequence_by=col("sequenceNum"),
    except_column_list=["sequenceNum"],      # overwrite all
    stored_as_scd_type= 2 ,       # <- this is the key difference!
)

/databricks/spark/python/dlt/helpers.py:139: UserWarning: Function create_target_table has been deprecated. Please use create_streaming_table instead.
  warnings.warn("Function {0} has been deprecated. Please use {1} instead."


Name,Type
id,int
name,string
price,"decimal(10,2)"
stock,int
